# ESG Media Sentiment Analysis (GDELT GKG)

**Raw dataset:** one 15-minute GDELT GKG batch (`20261117200000.gkg.csv`).

This notebook loads the raw GKG file, assigns the standard GDELT 2.0 GKG column headers, and displays the data.

## User Parameters

Edit the values in the cell below to control which data the pipeline processes. Then run all cells.

- **date_range**: inclusive `(start, end)` dates, ISO format (`"2026-11-17"`) or GDELT timestamp (`"20261117200000"`). Use `None` to leave a bound open.
- **continents**: e.g. `["North America", "Europe", "Asia"]`, or `["ALL"]` for no filter.
- **countries**: e.g. `["US", "GB", "ID"]` (ISO codes) or country names, or `["ALL"]` for no filter.
- **aggregation_freq**: `"1D"` (daily), `"1W"` (weekly), `"1M"` (monthly). Default `"1W"`.
- **min_articles**: minimum articles required for a (country, time) bucket to be colored on the map. Buckets below this are hidden as too noisy. Set `0` to disable.

In [1]:
# ============================================================
# USER PARAMETERS - edit these to control the pipeline
# ============================================================

# Inclusive date range (start, end). None = no bound.
# ISO format "2026-11-17" or GDELT timestamp "20261117200000".
date_range = ("20241117200000", "20251117200000")

# Target continents, e.g. ["North America", "Europe", "Asia"].
# Use ["ALL"] for no continent filter.
continents = ["Europe"]

# Target countries (ISO codes or names), e.g. ["US", "GB", "ID"].
# Use ["ALL"] for no country filter.
countries = ["ALL"]

# Aggregation frequency: "1D" daily, "1W" weekly, "1M" monthly.
aggregation_freq = "1M"

# Minimum articles required for a map bucket to be colored (0 = disable mask).
min_articles = 3

# Apply the user parameters to the config module
import config
config.date_range = date_range
config.continents = continents
config.countries = countries
config.aggregation_freq = aggregation_freq
config.MIN_ARTICLES = min_articles

print('date_range:', config.date_range)
print('continents:', config.continents)
print('countries:', config.countries)
print('aggregation_freq:', config.aggregation_freq)
print('min_articles:', config.MIN_ARTICLES)

date_range: ('20241117200000', '20251117200000')
continents: ['Europe']
countries: ['ALL']
aggregation_freq: 1M
min_articles: 3


In [2]:
# Download raw GDELT GKG batches based on the date_range above
# ============================================================
# DOWNLOAD SETTINGS - edit these to control downloading
# ============================================================
# Set to True to download data, False to skip (use existing files).
download_enabled = True

# Download mode:
#   'daily'  - one batch per day (23:45) in the date range
#   'all'    - every 15-minute batch in the date range (96/day)
#   'latest' - the most recent available batch (ignores date_range)
download_mode = 'daily'

# Set to True to re-download and overwrite files that already exist.
# False (default) keeps existing files and only downloads missing ones.
force_download = False

# Skip GDELT batches the server does not have (404) with a warning, instead
# of aborting. GDELT has occasional archive gaps (e.g. server outages).
# Set to False to stop on the first missing batch.
skip_missing = True

from download_raw import download_batch, download_date_range, download_all_batches, download_latest_batch

if download_enabled:
    if download_mode == 'daily':
        start, end = config.date_range
        if start is None or end is None:
            raise ValueError('download_mode=daily requires both date_range bounds (set them in the parameters cell)')
        paths = download_date_range(start, end, force=force_download, skip_missing=skip_missing)
    elif download_mode == 'all':
        start, end = config.date_range
        if start is None or end is None:
            raise ValueError('download_mode=all requires both date_range bounds (set them in the parameters cell)')
        paths = download_all_batches(start, end, force=force_download, skip_missing=skip_missing)
    elif download_mode == 'latest':
        paths = [download_latest_batch(force=force_download)]
    else:
        raise ValueError(f"Unknown download_mode: {download_mode!r} (use 'daily', 'all', or 'latest')")
    print('downloaded:', [p.name for p in paths])
else:
    print('download disabled - using existing raw files')

print('raw data dir:', config.raw_data_dir)
print('existing raw files:', [f.name for f in config.raw_data_dir.glob('*.gkg.csv')])


download: fetching https://data.gdeltproject.org/gdeltv2/20241117234500.gkg.csv.zip
download: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/raw/20241117234500.gkg.csv
download: fetching https://data.gdeltproject.org/gdeltv2/20241118234500.gkg.csv.zip
download: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/raw/20241118234500.gkg.csv
download: fetching https://data.gdeltproject.org/gdeltv2/20241119234500.gkg.csv.zip
download: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/raw/20241119234500.gkg.csv
download: fetching https://data.gdeltproject.org/gdeltv2/20241120234500.gkg.csv.zip
download: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/raw/20241120234500.gkg.csv
download: fetching https://data.gdeltproject.org/gdeltv2/20241121234500.gkg.csv.zip
download: saved /home/theodoret/Downloads/Interdiscplinarly/main/data/raw/20241121234500.gkg.csv
download: fetching https://data.gdeltproject.org/gdeltv2/20241122234500.gkg.csv.zip
download: s

HTTPError: 404 Client Error: Not Found for url: https://data.gdeltproject.org/gdeltv2/20250614234500.gkg.csv.zip

In [ ]:
import sys
from pathlib import Path

# Make config.py importable
sys.path.insert(0, str(Path.cwd()))

import config
import pandas as pd

print('raw dir:', config.raw_data_dir)

In [ ]:
# Locate and load the raw GKG files via the fetch_raw module
from fetch_raw import fetch_raw_gkg, find_raw_gkg_files

df, raw_paths = fetch_raw_gkg(date_range=config.date_range)
print('raw GKG files found:', [f.name for f in find_raw_gkg_files()])
print('loaded files:', len(raw_paths), '->', [p.name for p in raw_paths])
print('shape:', df.shape)
print('columns:', list(df.columns))


In [ ]:
# Preprocess: NA handling, dedup, and persistence via the preprocess module
from preprocess import preprocess, save_processed

df_clean = preprocess(df)
print('clean shape:', df_clean.shape)

clean_path = save_processed(df_clean)
print('saved to:', clean_path)

In [ ]:
# ESG classification: multi-label tagging via the esg_classify module
from esg_classify import classify

df_esg = classify(df_clean)
print('esg shape:', df_esg.shape)

pillar_cols = [f'is_{p}' for p in config.esg_categories]
print('tagged counts:')
print(df_esg[pillar_cols].sum())
print('multi-label rows:', (df_esg[pillar_cols].sum(axis=1) > 1).sum())
print('unclassified rows:', (df_esg[pillar_cols].sum(axis=1) == 0).sum())

In [ ]:
# Tone / sentiment extraction via the tone module
from tone import extract_tone

df_tone = extract_tone(df_esg)
print('tone shape:', df_tone.shape)

tone_cols = ['tone', 'positive', 'negative', 'polarity']
print('rows with tone:', df_tone['tone'].notna().sum())
print(df_tone[tone_cols].describe())

In [ ]:
# Media source geographic attribution via the geo_attr module
from geo_attr import attribute, filter_by_countries

df_geo = attribute(df_tone)
print('geo shape (before filter):', df_geo.shape)

# Apply user country / continent filters
df_geo = filter_by_countries(df_geo, countries=config.countries, continents=config.continents)
print('geo shape (after filter):', df_geo.shape)

print('source_country coverage:', df_geo['source_country'].notna().sum(), '/', len(df_geo))
print('location_country coverage:', df_geo['location_country'].notna().sum(), '/', len(df_geo))
print('attributed_country coverage:', (df_geo['attributed_country'] != config.UNKNOWN_SOURCE).sum(), '/', len(df_geo))
print('\ntop attributed countries:')
print(df_geo['attributed_country'].value_counts().head(10))

In [ ]:
# Metric aggregation: ESG x country x time window via the aggregate module
from aggregate import aggregate, save_aggregated

df_agg = aggregate(df_geo, freq=config.aggregation_freq)
print('agg shape:', df_agg.shape)
print('freq:', config.aggregation_freq)
print('categories:', sorted(df_agg['esg_category'].unique()))
print('countries:', df_agg['country'].nunique())
print('date range:', df_agg['date'].min(), '->', df_agg['date'].max())
print('\nhead:')
print(df_agg.head(10).to_string(index=False))

agg_path = save_aggregated(df_agg)
print('saved to:', agg_path)

In [ ]:
# Interactive world map visualization via the viz_map module
from viz_map import build_choropleth, build_date_slider_figure, build_category_selector_figure, build_time_series_figure, save_figure

# Latest-date choropleth for each ESG category
for cat in config.esg_categories + ['Composite']:
    fig = build_choropleth(df_agg, category=cat)
    save_figure(fig, f'map_{cat.lower()}')

# Date-slider figure (Composite)
slider_fig = build_date_slider_figure(df_agg, category='Composite')
save_figure(slider_fig, 'map_composite_slider')

# Category-switcher figure (dropdown: E / S / G / Composite)
selector_fig = build_category_selector_figure(df_agg)
save_figure(selector_fig, 'map_category_selector')

# Time-series figure: category dropdown + date slider (change over time)
time_fig = build_time_series_figure(df_agg)
save_figure(time_fig, 'map_time_series')

# Show the time-series map inline
build_time_series_figure(df_agg)

In [ ]:
# Show the data
df_esg.head()

In [ ]:
# Focus on the core columns from the plan
core = ['DATE', 'DocumentIdentifier', 'V2Themes', 'V2Locations', 'V2Tone']
df[core].head(10)